In [1]:
import numpy as np
from scipy.optimize import fsolve

In [2]:
def define_geometry():
    A1 = np.array([0, 0])
    A2 = np.array([0.2, -0.1])
    B1 = np.array([0, 0.2])
    B2 = np.array([0.2, 0.1])
    C = np.array([1, 0])
    T1 = np.array([0.5, 0.1])
    T2 = np.array([1.2, 0.1])

    L_A1C = np.linalg.norm(C - A1)
    L_A2C = np.linalg.norm(C - A2)
    L_B1D = np.linalg.norm(C - B1)  # Assuming D coincides with C for simplicity
    L_B2D = np.linalg.norm(C - B2)
    L_T1T2 = np.linalg.norm(T2 - T1)

    joint_positions = (A1, A2, B1, B2, C, T1, T2)
    arm_lengths = (L_A1C, L_A2C, L_B1D, L_B2D, L_T1T2)

    return joint_positions, arm_lengths


# Initialize geometry
joint_positions, arm_lengths = define_geometry()

# Display initial joint positions and arm lengths
joint_positions, arm_lengths

((array([0, 0]),
  array([ 0.2, -0.1]),
  array([0. , 0.2]),
  array([0.2, 0.1]),
  array([1, 0]),
  array([0.5, 0.1]),
  array([1.2, 0.1])),
 (1.0, 0.806225774829855, 1.019803902718557, 0.806225774829855, 0.7))

In [3]:
def kinematics(joint_positions, arm_lengths, travel):
    A1, A2, B1, B2, C, T1, T2 = joint_positions
    L_A1C, L_A2C, L_B1D, L_B2D, L_T1T2 = arm_lengths

    def equations(vars):
        Cx, Cy, T2x, T2y = vars
        C_new = np.array([Cx, Cy])
        T2_new = np.array([T2x, T2y])

        eq1 = np.linalg.norm(C_new - A1) - L_A1C
        eq2 = np.linalg.norm(C_new - A2) - L_A2C
        eq3 = np.linalg.norm(T2_new - T1) - L_T1T2
        eq4 = Cy - (C[1] + travel)  # Additional constraint for vertical travel

        return [eq1, eq2, eq3, eq4]

    # Initial guess for new positions
    initial_guess = [C[0], C[1] + travel, T2[0], T2[1]]
    solution = fsolve(equations, initial_guess)
    C_new = solution[0:2]
    T2_new = solution[2:4]

    return C_new, T2_new

# Example usage with travel of 0.1 units
travel = 0.1
C_new, T2_new = kinematics(joint_positions, arm_lengths, travel)

# Display new positions of C and T2
C_new, T2_new

c:\Users\brent\AppData\Local\Programs\Python\Python311\Lib\site-packages\scipy\optimize\_minpack_py.py:177: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


(array([1. , 0.1]), array([1.2, 0.1]))

In [4]:
def calculate_forces(joint_positions, C_new, T2_new, F_tire, theta_tire):
    A1, A2, B1, B2, C, T1, T2 = joint_positions

    # Define force vectors
    F_tire_vec = np.array([F_tire * np.sin(theta_tire), -F_tire * np.cos(theta_tire)])

    # Define unknown force vectors as numpy arrays
    F_A1 = np.array([0, 0])
    F_A2 = np.array([0, 0])
    F_B1 = np.array([0, 0])
    F_B2 = np.array([0, 0])
    F_T1 = np.array([0, 0])
    F_T2 = np.array([0, 0])

    # Create the system of linear equations for force equilibrium
    eqs = np.array([
        [1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0],  # Sum of Fx = 0
        [0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],  # Sum of Fy = 0
        [0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0],  # Sum of Fx = 0
        [0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0],  # Sum of Fy = 0
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],  # Sum of Fx = 0
        [0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]   # Sum of Fy = 0
    ])

    # Ensure the number of equations and unknowns match
    b = np.array([0, 0, 0, 0, F_tire_vec[0], F_tire_vec[1]])

    # Solve the system of equations
    solution = np.linalg.solve(eqs, b)
    F_A1[0], F_A1[1], F_A2[0], F_A2[1], F_B1[0], F_B1[1], F_B2[0], F_B2[1], F_T1[0], F_T1[1], F_T2[0], F_T2[1] = solution

    return F_A1, F_A2, F_B1, F_B2, F_T1, F_T2

# Example usage
F_tire = 1000  # Tire force in Newtons
theta_tire = np.deg2rad(-10)  # Tire force angle in radians

# Calculate forces
F_A1, F_A2, F_B1, F_B2, F_T1, F_T2 = calculate_forces(joint_positions, C_new, T2_new, F_tire, theta_tire)

# Display forces
F_A1, F_A2, F_B1, F_B2, F_T1, F_T2

LinAlgError: Last 2 dimensions of the array must be square

In [15]:
# Main function
def main():
    travel = 0.1
    F_tire = 1000
    theta_tire = np.deg2rad(-10)

    joint_positions, arm_lengths = define_geometry()
    C_new, T2_new = kinematics(joint_positions, arm_lengths, travel)
    F_A1, F_A2, F_B1, F_B2, F_T1, F_T2 = calculate_forces(*joint_positions, C_new, joint_positions[5], T2_new, F_tire, theta_tire)

    print("Forces at A1:", F_A1)
    print("Forces at A2:", F_A2)
    print("Forces at B1:", F_B1)
    print("Forces at B2:", F_B2)
    print("Forces at T1:", F_T1)
    print("Forces at T2:", F_T2)

if __name__ == "__main__":
    main()

TypeError: fsolve: there is a mismatch between the input and output shape of the 'func' argument 'equations'.Shape should be (4,) but it is (6,).